# 02 – Baseline Occurrence Rate
Simple counting occurrence rate (conventional approach).

In [1]:
import os
import glob
import pandas as pd
import numpy as np

# Kaggle input files are read-only. Set this manually if automatic detection
# finds the wrong CSV, for example:
# CSV_PATH = "/kaggle/input/YOUR-DATASET-SLUG/cumulative.csv"
CSV_PATH = None

if CSV_PATH is None:
    candidates = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    koi_candidates = [
        p for p in candidates
        if "cumulative" in os.path.basename(p).lower()
        or "koi" in os.path.basename(p).lower()
    ]
    print("Possible Kepler/KOI CSV files:")
    for p in koi_candidates[:30]:
        print(p)
    if len(koi_candidates) == 1:
        CSV_PATH = koi_candidates[0]
    elif len(koi_candidates) == 0:
        raise FileNotFoundError(
            "No likely Kepler/KOI CSV found. Attach the Kepler KOI dataset "
            "to this Kaggle notebook and set CSV_PATH manually."
        )
    else:
        raise ValueError(
            "More than one possible CSV was found. Set CSV_PATH manually "
            "to the correct cumulative KOI catalog."
        )

print("Using:", CSV_PATH)
df_raw = pd.read_csv(CSV_PATH, comment="#", low_memory=False)
print("Raw shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))

Possible Kepler/KOI CSV files:
/kaggle/input/datasets/sairakeshnagaram/koi-dataset/cumulative_2026.10.20_19.25.07.csv
Using: /kaggle/input/datasets/sairakeshnagaram/koi-dataset/cumulative_2026.10.20_19.25.07.csv
Raw shape: (9564, 49)
Columns: ['kepid', 'kepoi_name', 'kepler_name', 'koi_disposition', 'koi_pdisposition', 'koi_score', 'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec', 'koi_period', 'koi_period_err1', 'koi_period_err2', 'koi_time0bk', 'koi_time0bk_err1', 'koi_time0bk_err2', 'koi_impact', 'koi_impact_err1', 'koi_impact_err2', 'koi_duration', 'koi_duration_err1', 'koi_duration_err2', 'koi_depth', 'koi_depth_err1', 'koi_depth_err2', 'koi_prad', 'koi_prad_err1', 'koi_prad_err2', 'koi_teq', 'koi_teq_err1', 'koi_teq_err2', 'koi_insol', 'koi_insol_err1', 'koi_insol_err2', 'koi_model_snr', 'koi_tce_plnt_num', 'koi_tce_delivname', 'koi_steff', 'koi_steff_err1', 'koi_steff_err2', 'koi_slogg', 'koi_slogg_err1', 'koi_slogg_err2', 'koi_srad', 'koi_srad_err1', 'koi_srad

In [2]:
required = {"kepid", "koi_disposition", "koi_prad", "koi_steff"}
missing = required - set(df_raw.columns)
if missing:
    raise KeyError(f"Missing required columns: {sorted(missing)}")

df = df_raw.copy()
df["koi_prad"] = pd.to_numeric(df["koi_prad"], errors="coerce")
df["koi_steff"] = pd.to_numeric(df["koi_steff"], errors="coerce")
df["kepid"] = pd.to_numeric(df["kepid"], errors="coerce")

# Keep confirmed, false-positive, and candidate entries for the catalog sample.
allowed_dispositions = {"CONFIRMED", "FALSE POSITIVE", "CANDIDATE"}
df = df[df["koi_disposition"].isin(allowed_dispositions)]

# Match the main project's small-planet and FGK-like temperature selection.
df = df[
    df["kepid"].notna()
    & df["koi_prad"].between(1, 4, inclusive="both")
    & df["koi_steff"].between(4700, 6500, inclusive="both")
].copy()

df["kepid"] = df["kepid"].astype("int64")
print("Filtered KOI rows:", len(df))
print("Unique host stars:", df["kepid"].nunique())
print(df["koi_disposition"].value_counts(dropna=False))

Filtered KOI rows: 4249
Unique host stars: 3517
koi_disposition
CONFIRMED         1966
FALSE POSITIVE    1229
CANDIDATE         1054
Name: count, dtype: int64


In [3]:
import pandas as pd

# Load the stellar dataset using your exact Kaggle input path
stellar_csv_path = "/kaggle/input/datasets/sairakeshnagaram/koi-stellars/kepler_stellar_17.csv"
stellar_df = pd.read_csv(stellar_csv_path)

# Filter for sun-like stars with effective temperature (teff) between 4700 K and 6500 K
sun_like_stars = stellar_df[
    (stellar_df["teff"] >= 4700) & (stellar_df["teff"] <= 6500)
]

# Set the new denominator (total sun-like stars targeted by Kepler)
n_unique_stars = len(sun_like_stars["kepid"].dropna().unique())

# Count stars with >= 1 confirmed planet (numerator stays 1,463)
confirmed_host_ids = (
    df.loc[df["koi_disposition"].eq("CONFIRMED"), "kepid"].dropna().unique()
)
n_stars_with_confirmed_planet = len(confirmed_host_ids)

if n_unique_stars == 0:
    raise ValueError("No stars remain after filtering; check the dataset and filters.")

# Calculate baseline catalog fraction
baseline_catalog_fraction = n_stars_with_confirmed_planet / n_unique_stars

print(f"Unique sun-like stars target group:     {n_unique_stars:,}")
print(f"Stars with >=1 confirmed entry:          {n_stars_with_confirmed_planet:,}")
print(f"Baseline catalog fraction:                {baseline_catalog_fraction:.4f}")
print(f"Percentage:                               {100 * baseline_catalog_fraction:.2f}%")

Unique sun-like stars target group:     163,677
Stars with >=1 confirmed entry:          1,463
Baseline catalog fraction:                0.0089
Percentage:                               0.89%


In [4]:
summary = pd.DataFrame([{
    "n_filtered_koi_rows": len(df),
    "n_unique_stars": n_unique_stars,
    "n_stars_with_confirmed_planet": n_stars_with_confirmed_planet,
    "baseline_catalog_fraction": baseline_catalog_fraction,
    "baseline_catalog_fraction_percent": 100 * baseline_catalog_fraction,
    "radius_min_earth": 1,
    "radius_max_earth": 4,
    "stellar_temperature_min_K": 4700,
    "stellar_temperature_max_K": 6500
}])

output_path = "/kaggle/working/02_baseline_catalog_fraction.csv"
summary.to_csv(output_path, index=False)
print("Saved:", output_path)
display(summary)

Saved: /kaggle/working/02_baseline_catalog_fraction.csv


,n_filtered_koi_rows,n_unique_stars,n_stars_with_confirmed_planet,baseline_catalog_fraction,baseline_catalog_fraction_percent,radius_min_earth,radius_max_earth,stellar_temperature_min_K,stellar_temperature_max_K
0,4249,163677,1463,0.008938,0.893834,1,4,4700,6500
